In [1]:
import sys
from typing import Literal

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from scipy.stats import genpareto, lognorm, truncpareto

from pandemic_model.stats import fit_trunc_pareto

sys.path.append("../scripts")
from fit_distributions import fit_truncated_pareto

In [2]:
marani_raw = pd.read_excel("../../data/raw/epidemics_marani_240816.xlsx")

In [3]:
with open("../../data/clean/inverted_covid_severity.yaml", 'rb') as f:
  inverted_covid_severity_dict = yaml.safe_load(f)
  inverted_covid_severity = inverted_covid_severity_dict['ex_ante_severity']

In [4]:
marani_raw['intensity'] = marani_raw['severity_smu'] / marani_raw['duration']

#### Construct datasets

In [5]:
marani_classic_ds = marani_raw[marani_raw['year_end'].between(1600, 1945)]
all_modern_ds = marani_raw[marani_raw['year_start'] >= 1900]
modern_viral_ds = all_modern_ds[all_modern_ds['type'].str.contains('viral', case=False)]

In [6]:
modern_viral_ds['disease'].value_counts().sort_values(ascending=False)

disease
polio                         19
influenza                     18
smallpox                      13
meningitis                    11
measles                        9
dengue                         9
encephalitis                   8
yellow fever                   6
pneumonia                      5
murray valley encephalitis     4
ebola                          3
west nile                      2
rubella                        2
mumps                          1
kyasanur forest disease        1
hemorrhagic fever              1
rift valley fever              1
hiv/aids                       1
sars                           1
mers                           1
covid-19                       1
Name: count, dtype: int64

In [7]:
# Not sure about these
recurring_viruses = ['polio', 'smallpox', 'meningitis', 'measles', 'dengue',
                     'encephalitis', 'yellow fever', 'pneumonia', 'murray valley encephalitis',
                     'rubella', 'mumps']
non_resp_novel = ['ebola', 'west nile', 'kyasanur forest disease', 'hemorrhagic fever', 'rift valley fever', 'hiv/aids']

In [8]:
modern_viral_no_reccuring = modern_viral_ds[~modern_viral_ds['disease'].isin(recurring_viruses)]
modern_resp_novel = modern_viral_no_reccuring[~modern_viral_no_reccuring['disease'].isin(non_resp_novel)]

In [9]:
bernstein_intersect_ds = pd.read_excel("../../data/raw/novel_resp_241228.xlsx")

In [10]:
dfs = {
		'~Original Marani': marani_classic_ds,
		'Marani since 1900': all_modern_ds, 
		'Viral since 1900': modern_viral_ds,
		'Viral since 1900, no recurring': modern_viral_no_reccuring,
		'Respiratory since 1900': modern_resp_novel,
		'Bernstein novel': bernstein_intersect_ds
}

In [11]:
def get_pareto_dist(df,
                    col: Literal['severity_smu', 'intensity'],
                    disttype: Literal['gen', 'trunc'],
                    lower_bound=1e-2,
                    upper_bound=1e4):
		"""Get exceedance distribution."""
		fit_df = df.copy()
		
		# Drop observations below lower bound
		fit_df = fit_df[fit_df[col] >= lower_bound]
				
		if disttype == 'gen':
				c, loc, scale = genpareto.fit(fit_df[col], floc=lower_bound)
				dist = genpareto(c=c, loc=loc, scale=scale)                         
		else:
				(b, loc, scale), _ = fit_truncated_pareto(fit_df[col], lower_bound, upper_bound)
				c = (upper_bound - loc) / scale
				dist = truncpareto(b=b, c=c, loc=loc, scale=scale)

		return dist

In [12]:
def get_datasets_and_distributions(lower_bound=1e-2,
                                   drop_hiv=False,
                                   use_inverted_covid=False):
    """
    Creates plots comparing exceedance functions between generalized and truncated Pareto fits
    for both severity and intensity data.
    
    Args:
        lower_bound (float): Lower bound threshold for fitting distributions
        drop_hiv (bool): Whether to exclude HIV/AIDS observation
        use_inverted_covid (bool): Whether to use inverted COVID-19 severity
    
    Returns:
        dict: Dictionary containing the processed dataframes and fitted distributions
    """
    # Should probably define elsewhere
    upper_bounds = {
      'severity': {
        'gen': None,
        'trunc': 1e4
			},
      'intensity': {
        'gen': None,
        'trunc': 1e4
			}
		}
    
    def preprocess_df(df, drop_hiv=drop_hiv, use_inverted_covid=use_inverted_covid):
        new_df = df.copy()
        
        if drop_hiv:
            new_df = new_df[new_df['disease'] != 'hiv/aids']
        
        if use_inverted_covid:
            covid_mask = new_df['disease'] == 'covid-19'
            new_df.loc[covid_mask, 'severity_smu'] = inverted_covid_severity
        
        new_df['intensity'] = new_df['severity_smu'] / new_df['duration']
        return new_df
  
    # Process all dataframes
    processed_dfs = {name: preprocess_df(df) for name, df in dfs.items()}
    
    # Fit distributions
    distributions = {}
    for name, df in processed_dfs.items():
        distributions[name] = {
            metric: {
                dist_type: get_pareto_dist(
                    df,
                    'severity_smu' if metric == 'severity' else 'intensity',
                    dist_type,
                    lower_bound,
                    upper_bounds[metric][dist_type]
                )
                for dist_type in ['gen', 'trunc']
            }
            for metric in ['severity', 'intensity']
        }
    
    return {'dfs': processed_dfs, 'distributions': distributions}

In [13]:
# Calculate arrival rates
def get_recent_arrival_rate(df, year_start=2000, year_end=2021):
    return len(df[df['year_start'].between(year_start, year_end)]) / (year_end - year_start + 1)

recent_arrival_rates = {name: get_recent_arrival_rate(df) for name, df in dfs.items() if name != '~Original Marani'}
recent_arrival_rates['~Original Marani'] = 0.35

long_arrival_rates = {name: len(df) / 120 for name, df in dfs.items() if name != '~Original Marani'}
long_arrival_rates['~Original Marani'] = len(marani_classic_ds) / (1945 - 1600 + 1)

In [14]:
len(marani_classic_ds)

398

In [15]:
def get_share_above_threshold(df, col, thresh):
    return (
        len(df[df[col] >= thresh]) / len(df)
        if len(df) > 0 
        else 0
    )

In [17]:
# Interactive plotting code
def create_interactive_plot():
    """Creates and displays the interactive plot with all controls."""
    
    def update_plot(lower_bound, show_gen, show_trunc, use_recent_arrival_rate, use_above_threshold_rate, selected_dfs, use_extinction_upper, drop_hiv, use_inverted_covid):
        """Updates and returns the plot based on the selected parameters.
        
        Args:
            lower_bound (float): Lower bound for fitting distributions
            show_gen (bool): Whether to show generalized Pareto distribution curves
            show_trunc (bool): Whether to show truncated distribution curves 
            use_recent_arrival_rate (bool): Whether to use recent arrival rates
            use_above_threshold_rate (bool): Whether to adjust rates for above threshold events
            selected_dfs (list): List of selected datasets to plot
            use_extinction_upper (bool): Whether to use extinction upper bounds
            drop_hiv (bool): Whether to exclude HIV data
            use_inverted_covid (bool): Whether to use inverted COVID data
            
        Returns:
            matplotlib.figure.Figure: The updated figure
        """
        # Get initial datasets and distributions
        out = get_datasets_and_distributions(
						lower_bound=lower_bound,
						drop_hiv=drop_hiv,
						use_inverted_covid=use_inverted_covid
				)
        processed_dfs = out['dfs']
        distributions = out['distributions']

        fig, (ax_sev, ax_int) = plt.subplots(1, 2, figsize=(16, 8))
        
        # Generate points for theoretical curves
        x = np.logspace(np.log10(lower_bound), np.log10(1e4), 1000)
        
        for name in selected_dfs:
            df = processed_dfs[name]
            
            # Plot for both severity and intensity
            for ax, metric in [(ax_sev, 'severity'), (ax_int, 'intensity')]:
                
                y_start = recent_arrival_rates[name] if use_recent_arrival_rate else long_arrival_rates[name]
                
                if use_above_threshold_rate:
                    col = 'severity_smu' if metric == 'severity' else 'intensity'
                    share_above_threshold = get_share_above_threshold(df, col, lower_bound)
                    y_start = y_start * share_above_threshold
                            
                if show_gen:
                    y_gen = distributions[name][metric]['gen'].sf(x) * y_start
                    # Set to zero above upper bound
                    max_level = 171 if metric == 'severity' else 57
                    max_level = 1e4 if use_extinction_upper else max_level
                    y_gen[x > max_level] = 0
                    ax.semilogx(x, y_gen, label=f'{name} (GPD)', linestyle='-')
                
                if show_trunc:
                    y_trunc = distributions[name][metric]['trunc'].sf(x) * y_start
                    ax.semilogx(x, y_trunc, label=f'{name} (Truncated)', linestyle='--')
        
        # Configure axes
        ax_int.legend(loc='upper right')

        for ax, title in [(ax_sev, 'Severity'), (ax_int, 'Intensity')]:
            ax.set_ylim(0, 0.4)
            ax.set_xlim(1e-3, 1e4)
            ax.set_xlabel(f'{title} (deaths per 10k)')
            ax.set_ylabel('Exceedance probability (per year)')
            ax.grid(True, which='both', ls='-', alpha=0.2)
        
        plt.tight_layout()
        return fig
    
    # Create widgets
    lower_bound_slider = widgets.FloatLogSlider(
        value=1e-2, min=-3, max=1, description='Lower bound:'
    )
    
    show_gen = widgets.Checkbox(
        value=True, description='Show GPD'
    )
    
    show_trunc = widgets.Checkbox(
        value=True, description='Show Truncated'
    )
    
    use_recent_rate = widgets.Checkbox(
        value=True, description='Use recent arrival rates'
    )
    
    use_above_threshold = widgets.Checkbox(
        value=False, description='Adjust for above threshold reate'
    )
    
    use_extinction_upper = widgets.Checkbox(
        value=False, description='Use extinction upper bounds'
    )
    
    drop_hiv = widgets.Checkbox(
        value=False, description='Drop HIV'
    )
    
    use_inverted_covid = widgets.Checkbox(
        value=False, description='Use inverted COVID'
    )
    
    df_toggles = widgets.SelectMultiple(
        options=[
            '~Original Marani',
            'Marani since 1900',
            'Viral since 1900',
            'Viral since 1900, no recurring',
            'Respiratory since 1900',
            'Bernstein novel'
        ],
        value=['~Original Marani'],
        description='Datasets:'
    )
    
    # Create interactive plot
    interactive_plot = widgets.interactive(
        update_plot,
        lower_bound=lower_bound_slider,
        show_gen=show_gen,
        show_trunc=show_trunc,
        use_recent_arrival_rate=use_recent_rate,
        use_above_threshold_rate=use_above_threshold,
        selected_dfs=df_toggles,
        use_extinction_upper=use_extinction_upper,
        drop_hiv=drop_hiv,
        use_inverted_covid=use_inverted_covid
    )
    
    # Layout widgets and plot
    controls = widgets.VBox([
        df_toggles,
        widgets.HBox([lower_bound_slider, show_gen, show_trunc, use_recent_rate]),
        widgets.HBox([use_above_threshold, use_extinction_upper, drop_hiv, use_inverted_covid])
    ])
    
    return widgets.VBox([controls, interactive_plot.children[-1]])

# Display the interactive plot
create_interactive_plot()

- Need to use metastatistical extreme value distribution
- Need to make distribution fitting work at lower values